# Stock Price Prediction with a Deep Neural Network

## Goal

Your goal is to build a supervised learning model that predicts the next stock closing price of a company using the previous 20 closing prices.

You will use historical daily stock prices for **Apple Inc. (AAPL)**. The dataset is available as:

```text
data/aapl_daily.csv
```

The task is to convert a raw stock price time series into many training examples of the form:

```text
20 previous closing prices -> 21st closing price
```

Then you will build a deep neural network that takes 20 numeric inputs and predicts 1 numeric output.

# Solutions:
1. AVG
    - last 10 days
    - last 20 days
    - last 1 days
2. Linear Regression
    - last 2 points
    - last 3 point
    - last 10 points
3. AVG of AVGs
4. Taking periodicity into account
    - Picking every Monday for example. Picking the avg of the weeks
5. Impact of news / events
    - Augment the data with embeddings of news and events
*6. Building the Neural Network with 20 inputs and 1 output
7. Combine 4, 5 & 6 
8. RNN
9. Attention / Transformers


## Dataset

The dataset contains daily stock prices for **Apple Inc.**, ticker symbol **AAPL**.

Columns in the CSV file:

| Column | Meaning |
|---|---|
| Date | Trading date |
| Open | Opening stock price for that day |
| High | Highest stock price during that day |
| Low | Lowest stock price during that day |
| Close | Closing stock price for that day |
| Adj Close | Adjusted closing price |
| Volume | Number of shares traded |

For this exercise, use only the `Date` and `Close` columns. The `Close` column is the target time series.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# If TensorFlow is not installed in your environment, install it first.
# Example: pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print('TensorFlow version:', tf.__version__)

## Step 1: Load the Data

Load the CSV file and inspect the first few rows.

In [ ]:
data_path = 'data/aapl_daily.csv'
df = pd.read_csv(data_path)

df.head()

In [ ]:
df.info()
print('Number of rows:', len(df))
print('Date range:', df['Date'].min(), 'to', df['Date'].max())

## Step 2: Extract the Closing Prices

Use the `Close` column as the time series.

The full price sequence can be written as:

```text
p1, p2, p3, p4, ..., pn
```

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
close_prices = df['Close'].values.astype(float)

print(close_prices[:10])
print('Total closing prices:', len(close_prices))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], close_prices)
plt.title('AAPL Daily Closing Price')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.grid(True)
plt.show()

## Step 3: Convert the Time Series into Supervised Examples

Create a sliding-window dataset.

For every valid position `i`, create:

```text
X[i] = [Close[i], Close[i+1], ..., Close[i+19]]
y[i] = Close[i+20]
```

Each input has **20 values**, and the target is the **21st value**.

If the original dataset has `n` prices, the transformed dataset should have `n - 20` examples.

In [ ]:
def create_sliding_windows(series, window_size=20):
    X = []
    y = []
    
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size])
        y.append(series[i + window_size])
    
    return np.array(X), np.array(y)

window_size = 20
X, y = create_sliding_windows(close_prices, window_size)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('First input sequence:', X[0])
print('First target value:', y[0])

## Step 4: Train/Test Split

Because this is time-series data, do **not** randomly shuffle the data before splitting.

Use the earlier data for training and the later data for testing.

Suggested split:

```text
80% training
20% testing
```

In [ ]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

## Step 5: Scale the Data

Neural networks usually train better when numeric inputs are scaled.

Important rule: fit the scaler only on the training data, then use the same scaler to transform the test data. This prevents future information from leaking into the training process.

In [ ]:
# Scale input sequences
X_scaler = MinMaxScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

# Scale target values separately
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print('Scaled X_train min/max:', X_train_scaled.min(), X_train_scaled.max())
print('Scaled y_train min/max:', y_train_scaled.min(), y_train_scaled.max())

## Step 6: Build the Deep Neural Network

Build a feed-forward deep neural network, also called a multilayer perceptron.

Required model behavior:

```text
Input: 20 previous closing prices
Output: 1 predicted next closing price
```

Suggested architecture:

```text
Dense layer: 64 neurons, ReLU activation
Dense layer: 32 neurons, ReLU activation
Dense layer: 16 neurons, ReLU activation
Output layer: 1 neuron, linear activation
```

The final layer should not use softmax or sigmoid because this is a regression problem.

In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(20,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

## Step 7: Train the Model

Suggested starting values:

```text
epochs = 50
batch_size = 32
validation_split = 0.1
```

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

## Step 8: Evaluate the Model

Evaluate the model on the test set. Report at least:

- MAE: Mean Absolute Error
- RMSE: Root Mean Squared Error

Predictions should be converted back to the original price scale before calculating metrics.

In [ ]:
y_pred_scaled = model.predict(X_test_scaled)
y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('Test MAE:', mae)
print('Test RMSE:', rmse)

## Step 9: Plot Actual vs Predicted Prices

Create a visual comparison of the actual closing prices and predicted closing prices on the test set.

In [ ]:
test_dates = df['Date'].iloc[window_size + train_size:].values

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test, label='Actual Close')
plt.plot(test_dates, y_pred, label='Predicted Close')
plt.title('AAPL Actual vs Predicted Closing Prices')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)
plt.show()

## Questions to Answer

Answer these after training your model:

1. What is the shape of your `X` dataset after creating sliding windows?
2. What is the shape of your `y` dataset?
3. Why should the time-series train/test split not be randomly shuffled?
4. What loss function did you use, and why?
5. What was your test MAE?
6. What was your test RMSE?
7. Does the predicted line follow the real stock price trend?
8. Is predicting stock prices from only the previous 20 prices reliable in real life? Why or why not?

## Optional Extensions

After completing the basic version, try one or more of these improvements:

1. Use `Adj Close` instead of `Close`.
2. Predict percentage return instead of raw price.
3. Compare window sizes such as 5, 10, 20, 50, and 100.
4. Add more input features such as `Open`, `High`, `Low`, and `Volume`.
5. Compare the neural network against a naive baseline: tomorrow's predicted price = today's closing price.
6. Try an LSTM or 1D CNN model after finishing the simple dense network.

## Success Criteria

Your solution is successful if:

1. You correctly create sliding-window examples using 20 previous closing prices.
2. Your neural network accepts 20 inputs and produces 1 output.
3. You train only on earlier dates and test on later dates.
4. You report meaningful test metrics.
5. You visualize actual vs predicted prices.

The most important part is not achieving perfect predictions. The goal is to learn how to convert time-series data into a supervised learning problem and train a neural network for regression.